In [23]:
%load_ext autoreload
%autoreload 2

In [24]:
import os
import re
import shutil
import random
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from yolo_tools import get_yolo_label_df

In [34]:
data_dir = r'/localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002'
image_dir = os.path.join(data_dir, 'images')
label_dir = os.path.join(data_dir, 'labels')

bd_dir = r'/localnvme/data/billboard/bd_data/data687_mseg_c6_0917'

defect_list = ['deformation', 'broken', 'abandonment', 'corrosion']

In [26]:
def get_stem2img_dict(img_dir):
    img_list = [img_name for img_name in os.listdir(img_dir)]
    stem_list = [Path(img).stem for img in img_list]
    stem2img_dict = dict(zip(stem_list, img_list))
    return stem2img_dict
def find_defect(label_dir, image_dir, defect_list, exclude_dir=None):
    defect_file_list = []
    exclude_list = os.listdir(exclude_dir) if exclude_dir else None
    label_file_list = os.listdir(label_dir)
    stem2img_dict = get_stem2img_dict(image_dir)
    for label_name in tqdm(label_file_list):
        input_label_path = os.path.join(label_dir, label_name)
        df = get_yolo_label_df(input_label_path, mdet=True, attributes=defect_list)
        with_defect = (df[defect_list] > 0).any().any()
        if with_defect:
            image_name = stem2img_dict[Path(label_name).stem]
            if exclude_list is not None and image_name in exclude_list:
                continue
            defect_file_list.append(image_name)
    return defect_file_list

In [27]:
defect_file_list = find_defect(label_dir, image_dir, defect_list, exclude_dir=bd_dir)

100%|██████████| 7631/7631 [00:44<00:00, 172.48it/s]


In [28]:
len(defect_file_list)

2293

In [39]:
def random_select_defect(data_dir, defect_file_list, save_dir=None, train_ratio=0.9, random_seed=1010, full_path=True, suffix=''):
    image_dir = os.path.join(data_dir, 'images')
    label_dir = os.path.join(data_dir, 'labels')
    file_list = os.listdir(image_dir)
    if label_dir is not None:
        label_list = os.listdir(label_dir)
        label_list = [Path(label_name).stem for label_name in label_list]
        file_list_check = []
        for img_name in tqdm(file_list, desc='img check', total=len(file_list)):
            name = Path(img_name).stem
            if name in label_list:
                file_list_check.append(img_name)
        file_list = file_list_check
    if save_dir is None:
        save_dir = os.path.dirname(image_dir)

    np.random.seed(random_seed)
    np.random.shuffle(file_list)
    val_num = int(len(defect_file_list)*(1-train_ratio))

    val_list = defect_file_list[:val_num]
    train_list = [file_name for file_name in file_list if file_name not in val_list]

    if full_path:
        train_list = [os.path.join(image_dir, name) for name in train_list]
        val_list = [os.path.join(image_dir, name) for name in val_list]

    df_train = pd.DataFrame({'filename': train_list})
    df_val = pd.DataFrame({'filename': val_list})
    df_all = pd.DataFrame({'filename': train_list+val_list})
    df_train.to_csv(os.path.join(save_dir, f'train{suffix}.txt'), header=None, index=None)
    df_val.to_csv(os.path.join(save_dir, f'val{suffix}.txt'), header=None, index=None)
    df_all.to_csv(os.path.join(save_dir, 'all.txt'), header=None, index=None)
    print('%d save to %s,\n%d save to %s!'%(len(train_list), os.path.join(save_dir, f'train{suffix}.txt'),
                                           len(val_list), os.path.join(save_dir, f'val{suffix}.txt')))

In [40]:
random_select_defect(data_dir, defect_file_list, train_ratio=0.7, random_seed=1010, full_path=True, suffix='_70p')
random_select_defect(data_dir, defect_file_list, train_ratio=0.6, random_seed=1010, full_path=True, suffix='_60p')

img check: 100%|██████████| 7631/7631 [00:00<00:00, 23276.64it/s]


6944 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_70p.txt,
687 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_70p.txt!


img check: 100%|██████████| 7631/7631 [00:00<00:00, 21179.25it/s]


6714 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_60p.txt,
917 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_60p.txt!


In [41]:
def random_select_defect_ref(data_dir, ref_list, defect_file_list, save_dir=None, train_ratio=0.9, random_seed=1010, full_path=True, suffix=''):
    image_dir = os.path.join(data_dir, 'images')
    label_dir = os.path.join(data_dir, 'labels')
    file_list = os.listdir(image_dir)
    if label_dir is not None:
        label_list = os.listdir(label_dir)
        label_list = [Path(label_name).stem for label_name in label_list]
        file_list_check = []
        for img_name in tqdm(file_list, desc='img check', total=len(file_list)):
            name = Path(img_name).stem
            if name in label_list:
                file_list_check.append(img_name)
        file_list = file_list_check
    if save_dir is None:
        save_dir = os.path.dirname(image_dir)
    np.random.seed(random_seed)
    np.random.shuffle(file_list)
    val_num = int(len(defect_file_list)*(1-train_ratio))


    intersection_list = list(set(defect_file_list) & set(ref_list))
    remain_list = [file_name for file_name in defect_file_list if file_name not in intersection_list]
    val_select_num = max(0, val_num - len(intersection_list))
    val_select_list = random.sample(remain_list, min(val_select_num, len(remain_list)))

    val_list = val_select_list + intersection_list
    train_list = [file_name for file_name in file_list if file_name not in val_list]

    if full_path:
        train_list = [os.path.join(image_dir, name) for name in train_list]
        val_list = [os.path.join(image_dir, name) for name in val_list]

    df_train = pd.DataFrame({'filename': train_list})
    df_val = pd.DataFrame({'filename': val_list})
    df_all = pd.DataFrame({'filename': train_list+val_list})
    df_train.to_csv(os.path.join(save_dir, f'train{suffix}.txt'), header=None, index=None)
    df_val.to_csv(os.path.join(save_dir, f'val{suffix}.txt'), header=None, index=None)
    df_all.to_csv(os.path.join(save_dir, 'all.txt'), header=None, index=None)
    print('%d save to %s,\n%d save to %s!'%(len(train_list), os.path.join(save_dir, f'train{suffix}.txt'),
                                           len(val_list), os.path.join(save_dir, f'val{suffix}.txt')))

In [42]:
val_80p_ref_path = r'/localnvme/data/billboard/fused_data/data7720_mseg_c6_1002/val_80p_ref.txt'
df = pd.read_csv(val_80p_ref_path, header=None, index_col=False, names=['file_path'])
file_path_list = df['file_path'].to_list()
file_name_list = [Path(file_path).name for file_path in file_path_list]


In [43]:
random_select_defect_ref(data_dir, file_name_list, defect_file_list, train_ratio=0.7, random_seed=1010, full_path=True, suffix='_70p_ref')
random_select_defect_ref(data_dir, file_name_list, defect_file_list, train_ratio=0.6, random_seed=1010, full_path=True, suffix='_60p_ref')

img check: 100%|██████████| 7631/7631 [00:00<00:00, 25417.58it/s]


6944 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_70p_ref.txt,
687 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_70p_ref.txt!


img check: 100%|██████████| 7631/7631 [00:00<00:00, 23138.51it/s]


6714 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_60p_ref.txt,
917 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_60p_ref.txt!
